[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/225/blob/main/08_data_cleaning/08_3_Type_Problems.ipynb)

# 08.3: Type Problems and Conversions

In module 06 you used `astype()` to convert a column to `"category"` and `pd.to_numeric(errors="coerce")` to handle a numeric column that was stored as text. Those were brief introductions. This notebook treats type problems as a first-class topic.

A column's dtype tells pandas what operations are valid on it. When the dtype does not match the logical meaning of the data, you get one of two outcomes: either an error you can see, or a silently wrong result you cannot. The silent failures are the dangerous ones.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
df = pd.read_csv(url)
df.columns = ["survived", "pclass", "name", "sex", "age", "sibsp", "parch", "fare"]
df.dtypes

survived      int64
pclass        int64
name            str
sex             str
age         float64
sibsp         int64
parch         int64
fare        float64
dtype: object

pandas read this CSV and inferred a dtype for each column. Those inferences are often correct, but they are based purely on what the values look like, not on what they mean.

## Problem 1: Numbers stored as strings

CSV files occasionally contain a numeric column where one row has a typo or a note instead of a number: `"twenty-two"`, `"N/A"`, or `"unknown"`. pandas reads the whole column as `object` because it cannot store strings and floats in the same column.

The symptom is that arithmetic silently fails. Let's create a small example to see what that looks like.

In [2]:
# Simulate a column where one entry is a text note instead of a number
ages_raw = pd.Series(["22", "35", "unknown", "41", "28", "N/A", "19"])
print("dtype:", ages_raw.dtype)
print(ages_raw)

dtype: str
0         22
1         35
2    unknown
3         41
4         28
5        N/A
6         19
dtype: str


The dtype is `object` because the column contains strings. Now watch what happens if you sort this column:

In [3]:
# Alphabetical sort on a string column silently gives the wrong order
print("Sorted as strings:", ages_raw.sort_values().tolist())

Sorted as strings: ['19', '22', '28', '35', '41', 'N/A', 'unknown']


Strings sort alphabetically, so `"19"` comes before `"22"` (which is correct), but `"41"` comes before `"N/A"` and `"unknown"` (which are not numbers at all). No error is raised. pandas sorted exactly what you asked it to sort: a column of strings.

The fix is `pd.to_numeric()` with `errors="coerce"`. The `coerce` setting converts anything it cannot parse as a number into `NaN` rather than raising an error.

In [4]:
ages_clean = pd.to_numeric(ages_raw, errors="coerce")
print("dtype:", ages_clean.dtype)
print(ages_clean)
print("NaN count:", ages_clean.isnull().sum())

dtype: float64
0    22.0
1    35.0
2     NaN
3    41.0
4    28.0
5     NaN
6    19.0
dtype: float64
NaN count: 2


The dtype is now `float64` (floats rather than integers because NaN cannot be represented in an integer column). The two non-numeric entries became `NaN`, which you can now count, visualize, and fill using the tools from notebook 08.2. Sorting now produces the correct numeric order.

In [5]:
print("Sorted numerically:", ages_clean.sort_values().tolist())

Sorted numerically: [19.0, 22.0, 28.0, 35.0, 41.0, nan, nan]


The NaN values sort to the end, and the remaining numbers are in the correct numeric order. Compare this to the string sort output: `"19"` came before `"22"` by coincidence before, but `"22"` would have come after `"19"` only if the strings happened to sort that way. Numeric sort is unambiguous.

## Problem 2: Booleans encoded as integers or text

The `survived` column is stored as `int64` with values `0` and `1`. This works for computing survival rates (`survived.mean()` gives the correct fraction). But what if a dataset stores the same information as `"yes"`/`"no"` or `"Y"`/`"N"`? Those are `object` columns, and arithmetic on them fails completely.

In [6]:
# Simulate a survived column stored as text
survived_text = df["survived"].map({1: "yes", 0: "no"})
print("dtype:", survived_text.dtype)
print(survived_text.head())

# This fails: cannot compute a mean of strings
try:
    print("mean:", survived_text.mean())
except TypeError as e:
    print("Error:", e)

dtype: str
0     no
1    yes
2    yes
3    yes
4     no
Name: survived, dtype: str
Error: Cannot perform reduction 'mean' with string dtype


The `mean()` call raised a `TypeError` because you cannot average strings. To fix it, map the text back to numbers using `.map()` or `.replace()`.

In [7]:
survived_bool = survived_text.map({"yes": 1, "no": 0})
print("dtype:", survived_bool.dtype)
print("survival rate:", survived_bool.mean().round(3))

dtype: int64
survival rate: 0.386


The `.map()` call with a dictionary replaces each value according to the mapping. The result is an `int64` column where arithmetic works correctly again. You can also convert directly to Python `bool` with `.astype(bool)` if you want `True`/`False` values rather than `1`/`0`.

## Problem 3: Categorical labels stored as integers

The `pclass` column stores passenger class as `1`, `2`, and `3`. These look like numbers, and pandas stores them as `int64`. But the numbers are labels, not measurements. Taking the average of `pclass` gives 2.3, which is not a real passenger class. Treating `pclass` as a continuous number is not meaningful.

In [8]:
# Arithmetic works but is meaningless
print("Mean pclass:", df["pclass"].mean().round(2))
print("dtype before:", df["pclass"].dtype)

Mean pclass: 2.31
dtype before: int64


A mean of 2.3 is a number, but it does not correspond to any real class. The `category` dtype communicates to pandas that this column has a fixed set of labels and arithmetic across them is not meaningful.

In [9]:
df["pclass"] = df["pclass"].astype("category")
print("dtype after:", df["pclass"].dtype)
print("categories:", df["pclass"].cat.categories.tolist())

dtype after: category
categories: [1, 2, 3]


The `cat.categories` attribute lists the allowed values for the column. `groupby()` still works on a `category` column, and so does `value_counts()`. What changes is that pandas knows the column represents a discrete set of labels, which enables memory optimizations and makes your intent explicit to anyone reading the code.

### Memory savings with `category`

For string columns with low cardinality (few unique values relative to total rows), converting to `category` can dramatically reduce memory usage. Instead of storing the full string in every row, pandas stores an integer code internally and maps it back to the string when needed.

In [10]:
sex_object = df["sex"]
sex_category = df["sex"].astype("category")

print("Memory as object:", sex_object.memory_usage(deep=True), "bytes")
print("Memory as category:", sex_category.memory_usage(deep=True), "bytes")
print("Unique values:", sex_object.nunique())

Memory as object: 47771 bytes
Memory as category: 1127 bytes
Unique values: 2


The `category` version uses a small fraction of the memory of the `object` version. For a dataset with hundreds of thousands of rows and a column that repeats the same few strings, this can reduce memory by 90% or more.

Use `category` when a column has a fixed, small set of values that you will group or filter on. Good candidates: `sex`, `pclass` (as a label), any status or code column. Poor candidates: `name`, `ticket`, or any column where nearly every value is unique.

## Problem 4: Dates stored as strings

The Titanic dataset does not have a date column, but almost every real dataset does. When a CSV is created, dates are almost always stored as strings. pandas reads them as `object`, and string dates cannot be sorted correctly across all formats, filtered by range, or subtracted to find durations.

`pd.to_datetime()` parses a string column into `datetime64` values. You will use this heavily in notebook 08.7. Here is a preview of what the conversion looks like:

In [11]:
date_strings = pd.Series(["2023-01-15", "2023-02-20", "2023-03-05"])
print("Before:", date_strings.dtype)

dates = pd.to_datetime(date_strings)
print("After: ", dates.dtype)
print(dates)

Before: str
After:  datetime64[us]
0   2023-01-15
1   2023-02-20
2   2023-03-05
dtype: datetime64[us]


The dtype is now `datetime64[ns]`. Once a column has this dtype, you can filter rows by date range (`df[df["date"] > "2023-02-01"]`), extract the month with `.dt.month`, and subtract two date columns to get a duration. Notebook 08.7 covers all of this.

## A type-conversion checklist

When you load a new dataset, check the dtypes immediately with `df.dtypes` and ask these questions for each column:

| The column contains... | Expected dtype | Tool if wrong |
|---|---|---|
| Numbers (some rows have text) | `float64` | `pd.to_numeric(errors="coerce")` |
| Yes/No or True/False labels | `bool` or `int64` | `.map({"yes": 1, "no": 0})` |
| A small fixed set of labels | `category` | `.astype("category")` |
| Dates or timestamps | `datetime64` | `pd.to_datetime()` |
| Arbitrary text | `object` | no change needed |

## What's next

The columns you have been working with so far contain numbers and simple labels. But many real datasets have richer text: names, addresses, product descriptions, notes. Notebook 08.4 introduces the `.str` accessor, pandas's built-in toolkit for cleaning and normalizing string columns.